# Sobol — model evaluations

Run SeapoPym for each Sobol parameter set and extract three quantities of
interest (mean biomass, variance, peak timing) at each of the six stations
(Manuscript Section 2.4.3).

Inputs: `data/sobol_samples.parquet`, `data/stations.zarr`.
Output: `data/sobol_results.parquet`.
Runtime (test mode): ~30 seconds. Production mode: 1.5-4 h on a workstation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import yaml
from dask.distributed import Client
from seapopym.configuration.no_transport import (
    ForcingParameter,
    ForcingUnit,
    FunctionalGroupParameter,
    FunctionalGroupUnit,
    FunctionalTypeParameter,
    MigratoryTypeParameter,
    NoTransportConfiguration,
)
from seapopym.model.no_transport_model import NoTransportModel


def _project_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Project root marker {marker!r} not found.")


PROJECT_ROOT = _project_root()
DATA_DIR = PROJECT_ROOT / "data"

with open(PROJECT_ROOT / "parameters.yaml") as f:
    SOBOL = yaml.safe_load(f)["sobol"]

PERIOD = SOBOL["analysis_period"]
METRICS = SOBOL["metrics"]

## Reshape station forcings into a compact (T, Y, X) grid

SeapoPym expects spatial forcings indexed by `(T, Y, X)`. Stations are arranged
on a `(Y=6, X=1)` grid sorted by latitude (station coordinates serve as `Y`
labels; the `X` axis is a single placeholder). This avoids NaN cells that
would otherwise propagate through the solver.

In [2]:
raw = xr.open_zarr(DATA_DIR / "stations.zarr").sel(
    time=slice(PERIOD["spin_up_start"], PERIOD["end"])
).load()

# Sort stations by latitude so the Y axis is monotonic (required by xarray.sel)
order = raw.station_lat.argsort().values
ordered = raw.isel(station=order)
y_values = ordered.station_lat.values.astype(np.float32)
x_values = np.array([0.0], dtype=np.float32)


def _to_grid(da: xr.DataArray) -> xr.DataArray:
    # (station, time) -> (T, Y=6, X=1)
    arr = da.values.T[:, :, np.newaxis]
    return xr.DataArray(
        arr,
        dims=("T", "Y", "X"),
        coords={"T": ordered.time.values, "Y": ("Y", y_values), "X": ("X", x_values)},
    )


# Temperature requires a Z dimension; the label must match day_layer/night_layer (0).
temperature = _to_grid(ordered.temperature).expand_dims(Z=[0], axis=1)
npp = _to_grid(ordered.npp)
for coord, axis in {"T": "T", "Z": "Z", "Y": "Y", "X": "X"}.items():
    if coord in temperature.coords:
        temperature[coord].attrs["axis"] = axis
    if coord in npp.coords:
        npp[coord].attrs["axis"] = axis
temperature.attrs["units"] = "degC"
npp.attrs["units"] = "mg/m2/day"

station_coords = pd.DataFrame(
    {"Y": y_values.astype(float), "X": np.zeros(len(y_values))},
    index=ordered.station.values,
)
station_coords

,Y,X
HOT,23.0,0.0
Canaries,30.0,0.0
BATS,32.0,0.0
Bay_of_Biscay,45.5,0.0
PAPA,50.0,0.0
BARENTS,75.0,0.0


## Distribute simulations across a Dask cluster

Each parameter set yields one SeapoPym simulation on the sparse grid.
Results are reduced to (mean, variance, argmax) at each station and
appended to a Parquet file batch by batch (resumable).

In [3]:
client = Client()
print(client.dashboard_link)

FORCING = ForcingParameter(
    temperature=ForcingUnit(forcing=temperature),
    primary_production=ForcingUnit(forcing=npp),
)
FORCING_FUTURE = client.scatter(FORCING, broadcast=True)

samples = pd.read_parquet(DATA_DIR / "sobol_samples.parquet")
columns = pd.MultiIndex.from_product(
    [station_coords.index, METRICS], names=["station", "metric"]
)

results_path = DATA_DIR / "sobol_results.parquet"
if results_path.exists():
    results = pd.read_parquet(results_path)
    print(f"Resuming: {len(results):,} / {len(samples):,} already computed.")
else:
    results = pd.DataFrame(columns=columns)

None unit is milligram / day / meter ** 2, it will be converted to gram / day / meter ** 2.


None unit is milligram / day / meter ** 2, it will be converted to gram / day / meter ** 2.


http://127.0.0.1:8787/status


In [4]:
BATCH_SIZE = 1000
T_ANALYSIS = slice(PERIOD["start"], PERIOD["end"])


def evaluate_sample(params, forcing_parameter):
    fg = FunctionalGroupParameter(functional_group=[FunctionalGroupUnit(
        name="zooplankton",
        energy_transfert=params[0],
        functional_type=FunctionalTypeParameter(
            lambda_temperature_0=params[3],
            gamma_lambda_temperature=params[4],
            tr_0=params[1],
            gamma_tr=params[2],
        ),
        migratory_type=MigratoryTypeParameter(day_layer=0, night_layer=0),
    )])
    config = NoTransportConfiguration(forcing=forcing_parameter, functional_group=fg)
    with NoTransportModel.from_configuration(config) as model:
        model.run()
        out = []
        for station, row in station_coords.iterrows():
            b = model.state.biomass.sel(
                T=T_ANALYSIS, Y=row["Y"], X=row["X"], functional_group=0,
            )
            # Extreme parameter combinations can yield all-NaN biomass; mask gracefully.
            try:
                argmax_val = int(b.argmax("T"))
            except ValueError:
                argmax_val = -1
            out.append([float(b.mean()), float(b.var()), argmax_val])
    return np.array(out).flatten()


n_batches = (len(samples) + BATCH_SIZE - 1) // BATCH_SIZE
for b in range(n_batches):
    lo, hi = b * BATCH_SIZE, min((b + 1) * BATCH_SIZE, len(samples))
    if hi - 1 in results.index:
        continue
    batch = samples.iloc[lo:hi]
    futures = client.map(evaluate_sample, batch.to_numpy(), forcing_parameter=FORCING_FUTURE)
    out = pd.DataFrame(client.gather(futures), columns=columns, index=batch.index)
    results = pd.concat([results, out])
    results.to_parquet(results_path)
    print(f"Batch {b+1}/{n_batches}: rows {lo}-{hi-1}")

client.close()
len(results)

Batch 1/13: rows 0-999


Batch 2/13: rows 1000-1999


Batch 3/13: rows 2000-2999


Batch 4/13: rows 3000-3999


Batch 5/13: rows 4000-4999


...

Batch 10/13: rows 9000-9999


Batch 11/13: rows 10000-10999


Batch 12/13: rows 11000-11999


Batch 13/13: rows 12000-12287


12288